# Explore SPECFEM2D model outputs

This notebook provides a compact workflow for checking Sarah/Glenn SPECFEM2D model outputs.

It can:

1. discover `Mod*/OUTPUT_FILES` directories,
2. load each gather, preferring binary SU files when readable and falling back to SPECFEM ASCII (`*.semv`/`*.semd`),
3. convert each gather to SEG-Y with basic geometry headers,
4. make wiggle plots for quick QC, and
5. difference two converted SEG-Y gathers trace-by-trace.

All generated products are written beneath `outdir`; nothing is intentionally saved in the code repository.


## 1. Configuration

Edit these paths for your machine. `repo_dir` should be the parent directory containing `segy_tools/` and `specfem_tools/`. `outdir` should be outside the code repository.


In [1]:
from pathlib import Path
import sys

# Parent directory containing segy_tools/ and specfem_tools/.
repo_dir = Path("~/Developer/karstgeo/01_modeling/specfem2d").expanduser()

# SPECFEM model root containing Mod10/OUTPUT_FILES, Mod12/OUTPUT_FILES, etc.
specfem_root = Path(
    "~/Library/CloudStorage/Box-Box/thompsong/"
    "2026KarstGeophysicsDEP/02_Modelling/Seismic/specfem2d"
).expanduser()

# All notebook-generated products go here, not into repo_dir.
outdir = specfem_root / "glenn"

# Default survey/model assumptions used only when headers do not contain usable coordinates.
receiver_spacing_m = 1.0
first_receiver_x_m = 0.0
source_x_m = 0.0

# Default plotting choices.
default_tmin = 0.0
default_tmax = 0.3
wiggle_scale = 0.02
normalize_wiggles = False

# Derived output directories.
fig_dir = outdir / "figures" / "explore_specfem_output"
segy_out_dir = outdir / "segy" / "specfem_models"
diff_fig_dir = outdir / "figures" / "specfem_model_differences"

for d in [fig_dir, segy_out_dir, diff_fig_dir]:
    d.mkdir(parents=True, exist_ok=True)

print(f"repo_dir      = {repo_dir}")
print(f"specfem_root  = {specfem_root}")
print(f"outdir        = {outdir}")
print(f"fig_dir       = {fig_dir}")
print(f"segy_out_dir  = {segy_out_dir}")


repo_dir      = /Users/glennthompson/Developer/karstgeo/01_modeling/specfem2d
specfem_root  = /Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/02_Modelling/Seismic/specfem2d
outdir        = /Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/02_Modelling/Seismic/specfem2d/glenn
fig_dir       = /Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/02_Modelling/Seismic/specfem2d/glenn/figures/explore_specfem_output
segy_out_dir  = /Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/02_Modelling/Seismic/specfem2d/glenn/segy/specfem_models


## 2. Imports

The notebook avoids `os.chdir()`. If the local packages have not been installed with `pip install -e .`, the repository path is added to `sys.path`.


In [2]:
if not (repo_dir / "segy_tools").exists():
    raise FileNotFoundError(
        f"Could not find segy_tools/ under repo_dir={repo_dir}. "
        "Set repo_dir to the parent directory of segy_tools/."
    )

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

import numpy as np
import pandas as pd
import obspy
from obspy import Stream, Trace, UTCDateTime
from obspy.core import AttribDict

from specfem_tools.io import read_sem_gather
from segy_tools.gather import plot_wiggle_gather_from_stream
from segy_tools.io import read_su_file, write_segy
from segy_tools.headers import make_trace_header
from segy_tools.plotting import plot_wiggle_gather, plot_difference_gathers

print("Imported local SPECFEM/SEG-Y tools")


Imported local SPECFEM/SEG-Y tools


## 3. Discover model output directories

Each model is expected to follow the pattern `Mod*/OUTPUT_FILES` under `specfem_root`.


In [3]:
def find_specfem_model_outputs(root):
    """Return sorted Mod*/OUTPUT_FILES directories under a SPECFEM root."""
    root = Path(root).expanduser()
    return sorted(p for p in root.glob("Mod*/OUTPUT_FILES") if p.is_dir())


def model_number_from_name(model_name):
    """Extract integer part from names like Mod17 or Mod13V. Returns None if absent."""
    digits = "".join(ch for ch in str(model_name) if ch.isdigit())
    return int(digits) if digits else None


model_output_dirs = find_specfem_model_outputs(specfem_root)

df_models = pd.DataFrame({
    "model": [p.parent.name for p in model_output_dirs],
    "model_number": [model_number_from_name(p.parent.name) for p in model_output_dirs],
    "output_dir": model_output_dirs,
})

df_models = df_models.sort_values(["model_number", "model"], na_position="last").reset_index(drop=True)
display(df_models)


,model,model_number,output_dir
0,Mod10,10,/Users/glennthompson/Library/CloudStorage/Box-...
1,Mod12,12,/Users/glennthompson/Library/CloudStorage/Box-...
2,Mod13,13,/Users/glennthompson/Library/CloudStorage/Box-...
3,Mod13V,13,/Users/glennthompson/Library/CloudStorage/Box-...
4,Mod14,14,/Users/glennthompson/Library/CloudStorage/Box-...
5,Mod16,16,/Users/glennthompson/Library/CloudStorage/Box-...
6,Mod17,17,/Users/glennthompson/Library/CloudStorage/Box-...


## 4. Load SPECFEM gathers

`load_specfem_gather()` prefers SU files because they are binary and faster to read. If no readable SU file is found, it falls back to SPECFEM ASCII files such as `AA.S0001.BXZ.semv`.

The function returns a dictionary. Use `specfem_gather_result_to_stream()` to convert either SU or SEM ASCII output into a standard ObsPy `Stream` with basic SEG-Y-ready headers.


In [4]:
def discover_su_files(output_dir):
    """Find candidate SU files in an OUTPUT_FILES directory."""
    return sorted(Path(output_dir).glob("*.su"))


def read_su_try_both_byteorders(path, dt_s=None, t0_s=0.0):
    """Read an SU file, trying little-endian first and then big-endian."""
    last_error = None
    for byteorder in ("<", ">"):
        try:
            st = read_su_file(path, dt_s=dt_s, t0_s=t0_s, byteorder=byteorder)
            return st, byteorder
        except Exception as exc:
            last_error = exc
    raise last_error


def load_specfem_gather(
    output_dir,
    component="BXZ",
    extension="semv",
    timing=None,
    prefer_su=True,
    verbose=True,
):
    """
    Load a SPECFEM gather from one OUTPUT_FILES directory.

    Parameters
    ----------
    output_dir : path-like
        A model's OUTPUT_FILES directory.
    component : str
        SPECFEM/SEGY component name to look for when falling back to ASCII,
        e.g. "BXZ" or "BXX".
    extension : str
        SPECFEM ASCII extension, usually "semv" or "semd".
    timing : optional
        Timing object passed through to read_sem_gather().
    prefer_su : bool
        If True, try *.su files before ASCII.
    verbose : bool
        Print status messages.

    Returns
    -------
    dict
        Either {"mode": "su", "stream": Stream, ...} or
        {"mode": "sem", "time": ..., "data": ..., "station_indices": ...}.
    """
    output_dir = Path(output_dir)

    if prefer_su:
        for sufile in discover_su_files(output_dir):
            try:
                st, byteorder = read_su_try_both_byteorders(sufile)
                if verbose:
                    print(f"Loaded SU: {sufile.name} byteorder={byteorder}")
                return {
                    "mode": "su",
                    "stream": st,
                    "path": sufile,
                    "component": component,
                    "byteorder": byteorder,
                }
            except Exception as exc:
                if verbose:
                    print(f"  SU failed: {sufile.name}: {type(exc).__name__}: {exc}")

    if verbose:
        print(f"Falling back to SPECFEM ASCII: component={component}, extension={extension}")

    time, data, station_indices = read_sem_gather(
        output_dir,
        component=component,
        extension=extension,
        timing=timing,
        verbose=verbose,
    )

    return {
        "mode": "sem",
        "time": time,
        "data": data,
        "station_indices": station_indices,
        "component": component,
        "extension": extension,
    }


In [5]:
def specfem_gather_result_to_stream(
    result,
    component="BXZ",
    receiver_spacing_m=1.0,
    first_receiver_x_m=0.0,
    source_x_m=0.0,
    shot_number=1,
    network="SY",
):
    """
    Convert load_specfem_gather() output into an ObsPy Stream.

    This standardizes both SU and SEM ASCII inputs into a single object that can
    be plotted, written to SEG-Y, or differenced against another model.
    """
    component = component.upper()

    if result["mode"] == "su":
        st = result["stream"].copy()

        for i, tr in enumerate(st, start=1):
            receiver_x_m = first_receiver_x_m + (i - 1) * receiver_spacing_m

            tr.stats.network = network
            tr.stats.station = f"S{i:04d}"
            tr.stats.location = f"{int(shot_number) % 100:02d}"
            tr.stats.channel = component

            tr.stats.segy = AttribDict()
            tr.stats.segy.trace_header = make_trace_header(
                station_idx=i,
                receiver_x_m=float(receiver_x_m),
                source_x_m=float(source_x_m),
                shot_number=int(shot_number),
                dt_s=float(tr.stats.delta),
                npts=int(tr.stats.npts),
                receiver_z_m=0.0,
                source_z_m=0.0,
            )

        return st

    if result["mode"] == "sem":
        time = np.asarray(result["time"], dtype=float)
        data = np.asarray(result["data"], dtype=np.float32)
        station_indices = np.asarray(result["station_indices"], dtype=int)

        dt = float(np.median(np.diff(time)))
        st = Stream()

        for i, station_index in enumerate(station_indices):
            receiver_x_m = first_receiver_x_m + (station_index - station_indices.min()) * receiver_spacing_m

            tr = Trace(data=data[i].astype(np.float32))
            tr.stats.network = network
            tr.stats.station = f"S{int(station_index):04d}"
            tr.stats.location = f"{int(shot_number) % 100:02d}"
            tr.stats.channel = component
            tr.stats.delta = dt
            tr.stats.starttime = UTCDateTime(1970, 1, 1) + float(time[0])

            tr.stats.segy = AttribDict()
            tr.stats.segy.trace_header = make_trace_header(
                station_idx=int(station_index),
                receiver_x_m=float(receiver_x_m),
                source_x_m=float(source_x_m),
                shot_number=int(shot_number),
                dt_s=dt,
                npts=int(tr.stats.npts),
                receiver_z_m=0.0,
                source_z_m=0.0,
            )

            st.append(tr)

        return st

    raise ValueError(f"Unknown result mode: {result['mode']}")


## 5. Convert and plot model gathers

This wrapper loads one model gather, converts it to an ObsPy `Stream`, optionally writes SEG-Y, and optionally writes a wiggle plot.


In [6]:
def model_name_from_output_dir(output_dir):
    """Return model name from a Mod*/OUTPUT_FILES path."""
    return Path(output_dir).parent.name


def load_model_as_stream(
    output_dir,
    component="BXZ",
    extension="semv",
    timing=None,
    prefer_su=True,
    receiver_spacing_m=1.0,
    first_receiver_x_m=0.0,
    source_x_m=0.0,
    shot_number=None,
    network="SY",
    verbose=True,
):
    """Load one model output directory and return a standardized ObsPy Stream."""
    output_dir = Path(output_dir)
    model_name = model_name_from_output_dir(output_dir)

    if shot_number is None:
        shot_number = model_number_from_name(model_name) or 1

    result = load_specfem_gather(
        output_dir,
        component=component,
        extension=extension,
        timing=timing,
        prefer_su=prefer_su,
        verbose=verbose,
    )

    st = specfem_gather_result_to_stream(
        result,
        component=component,
        receiver_spacing_m=receiver_spacing_m,
        first_receiver_x_m=first_receiver_x_m,
        source_x_m=source_x_m,
        shot_number=shot_number,
        network=network,
    )

    return st, result


def write_model_products(
    output_dir,
    component="BXZ",
    extension="semv",
    timing=None,
    prefer_su=True,
    receiver_spacing_m=1.0,
    first_receiver_x_m=0.0,
    source_x_m=0.0,
    write_segy_file=True,
    make_plot=True,
    tmin=0.0,
    tmax=0.3,
    scale=0.02,
    normalize=False,
    verbose=True,
):
    """Load one model, then write SEG-Y and/or a wiggle plot under outdir."""
    output_dir = Path(output_dir)
    model_name = model_name_from_output_dir(output_dir)
    shot_number = model_number_from_name(model_name) or 1

    st, result = load_model_as_stream(
        output_dir,
        component=component,
        extension=extension,
        timing=timing,
        prefer_su=prefer_su,
        receiver_spacing_m=receiver_spacing_m,
        first_receiver_x_m=first_receiver_x_m,
        source_x_m=source_x_m,
        shot_number=shot_number,
        verbose=verbose,
    )

    segy_file = segy_out_dir / f"{model_name}_{component}.segy"
    fig_file = fig_dir / f"{model_name}_{component}.png"

    if write_segy_file:
        segy_file.parent.mkdir(parents=True, exist_ok=True)
        write_segy(st, segy_file)
        if verbose:
            print(f"  wrote SEG-Y: {segy_file}")

    if make_plot:
        fig_file.parent.mkdir(parents=True, exist_ok=True)
        plot_wiggle_gather_from_stream(
            st,
            fallback_receiver_spacing_m=receiver_spacing_m,
            fallback_first_receiver_x_m=first_receiver_x_m,
            fallback_source_x_m=source_x_m,
            normalize=normalize,
            scale=scale,
            tmin=tmin,
            tmax=tmax,
            title=f"{model_name} {component}",
            outfile=fig_file,
        )
        if verbose:
            print(f"  wrote figure: {fig_file}")

    return st, result, segy_file, fig_file


In [7]:
# Example: process every discovered model for one component.
# Set verbose=False after initial debugging.

processed_rows = []

for output_dir in model_output_dirs:
    model_name = model_name_from_output_dir(output_dir)
    print(f"=== {model_name} ===")

    try:
        st, result, segy_file, fig_file = write_model_products(
            output_dir,
            component="BXZ",
            extension="semv",
            prefer_su=True,
            receiver_spacing_m=receiver_spacing_m,
            first_receiver_x_m=first_receiver_x_m,
            source_x_m=source_x_m,
            tmin=default_tmin,
            tmax=default_tmax,
            scale=wiggle_scale,
            normalize=normalize_wiggles,
            verbose=True,
        )
        print(f"  processed {model_name}: n_traces={len(st)}, segy_file={segy_file}, fig_file={fig_file}")

        processed_rows.append({
            "model": model_name,
            "component": "BXZ",
            "mode": result["mode"],
            "n_traces": len(st),
            "segy_file": segy_file,
            "figure_file": fig_file,
        })

    except Exception as exc:
        print(f"  skipped {model_name}: {type(exc).__name__}: {exc}")
        processed_rows.append({
            "model": model_name,
            "component": "BXZ",
            "mode": "failed",
            "n_traces": 0,
            "segy_file": None,
            "figure_file": None,
            "error": str(exc),
        })

processed_df = pd.DataFrame(processed_rows)
display(processed_df)


=== Mod10 ===
Loaded SU: BXX_velocity.su byteorder=>
  wrote SEG-Y: /Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/02_Modelling/Seismic/specfem2d/glenn/segy/specfem_models/Mod10_BXZ.segy
  wrote figure: /Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/02_Modelling/Seismic/specfem2d/glenn/figures/explore_specfem_output/Mod10_BXZ.png
  processed Mod10: n_traces=299, segy_file=/Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/02_Modelling/Seismic/specfem2d/glenn/segy/specfem_models/Mod10_BXZ.segy, fig_file=/Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/02_Modelling/Seismic/specfem2d/glenn/figures/explore_specfem_output/Mod10_BXZ.png
=== Mod12 ===
Falling back to SPECFEM ASCII: component=BXZ, extension=semv
Found 299 files in /Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/02_Modelling/Seismic/specfem2d/Mod12/OUTPUT_FI

,model,component,mode,n_traces,segy_file,figure_file,error
0,Mod10,BXZ,su,299,/Users/glennthompson/Library/CloudStorage/Box-...,/Users/glennthompson/Library/CloudStorage/Box-...,NaN
1,Mod12,BXZ,sem,299,/Users/glennthompson/Library/CloudStorage/Box-...,/Users/glennthompson/Library/CloudStorage/Box-...,NaN
2,Mod13,BXZ,sem,299,/Users/glennthompson/Library/CloudStorage/Box-...,/Users/glennthompson/Library/CloudStorage/Box-...,NaN
3,Mod13V,BXZ,failed,0,None,None,No SPECFEM ASCII files found in /Users/glennth...
4,Mod14,BXZ,failed,0,None,None,No SPECFEM ASCII files found in /Users/glennth...
5,Mod16,BXZ,sem,299,/Users/glennthompson/Library/CloudStorage/Box-...,/Users/glennthompson/Library/CloudStorage/Box-...,NaN
6,Mod17,BXZ,sem,299,/Users/glennthompson/Library/CloudStorage/Box-...,/Users/glennthompson/Library/CloudStorage/Box-...,NaN


## 6. Plot an individual SEG-Y or SU file

These cells are optional single-file checks. They are useful when debugging one file before running the batch processor.


In [12]:
def plot_segy_file(
    segy_file,
    outfile=None,
    tmin=0.0,
    tmax=0.3,
    receiver_spacing_m=1.0,
    first_receiver_x_m=0.0,
    source_x_m=0.0,
    scale=0.02,
    normalize=False,
):
    """Read and plot one SEG-Y file."""
    segy_file = Path(segy_file)
    st = obspy.read(str(segy_file), format="SEGY", unpack_trace_headers=True)

    if outfile is None:
        outfile = fig_dir / f"{segy_file.stem}_wiggle.png"

    plot_wiggle_gather_from_stream(
        st,
        fallback_receiver_spacing_m=receiver_spacing_m,
        fallback_first_receiver_x_m=first_receiver_x_m,
        fallback_source_x_m=source_x_m,
        tmin=tmin,
        tmax=tmax,
        scale=scale,
        normalize=normalize,
        title=f"{segy_file.name}: wiggle gather",
        outfile=outfile,
    )
    print(f"  plotted {segy_file} to {outfile}")

    return st


# Example:
st = plot_segy_file(segy_out_dir / "Mod17_BXZ.segy")


  plotted /Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/02_Modelling/Seismic/specfem2d/glenn/segy/specfem_models/Mod17_BXZ.segy to /Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/02_Modelling/Seismic/specfem2d/glenn/figures/explore_specfem_output/Mod17_BXZ_wiggle.png


In [13]:
def plot_su_directory(
    su_dir,
    pattern="*_*.su",
    tmin=-0.05,
    tmax=0.25,
    receiver_spacing_m=2.0,
    first_receiver_x_m=0.0,
    source_x_m=0.0,
    scale=0.02,
    normalize=False,
):
    """Read all matching SU files in a directory and write wiggle plots under outdir."""
    su_dir = Path(su_dir).expanduser()
    su_fig_dir = fig_dir / "su_wiggles"
    su_fig_dir.mkdir(parents=True, exist_ok=True)

    su_files = sorted(su_dir.glob(pattern))
    print(f"Found {len(su_files)} SU files in {su_dir}")

    rows = []
    for su_file in su_files:
        print(su_file)
        try:
            st, byteorder = read_su_try_both_byteorders(su_file)
            outfile = su_fig_dir / f"{su_file.stem}_wiggle.png"
            plot_wiggle_gather_from_stream(
                st,
                fallback_receiver_spacing_m=receiver_spacing_m,
                fallback_first_receiver_x_m=first_receiver_x_m,
                fallback_source_x_m=source_x_m,
                tmin=tmin,
                tmax=tmax,
                scale=scale,
                normalize=normalize,
                title=f"{su_file.stem}: SU wiggle gather",
                outfile=outfile,
            )
            print(f"  plotted {su_file} to {outfile}")
            rows.append({"file": su_file, "byteorder": byteorder, "n_traces": len(st), "figure_file": outfile})
        except Exception as exc:
            print(f"  skipping {su_file.name}: {type(exc).__name__}: {exc}")
            rows.append({"file": su_file, "byteorder": None, "n_traces": 0, "figure_file": None, "error": str(exc)})

    return pd.DataFrame(rows)


# Example:
su_df = plot_su_directory(Path("~/Downloads").expanduser())
display(su_df)


Found 2 SU files in /Users/glennthompson/Downloads
/Users/glennthompson/Downloads/Ux_file_single_v.su
  plotted /Users/glennthompson/Downloads/Ux_file_single_v.su to /Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/02_Modelling/Seismic/specfem2d/glenn/figures/explore_specfem_output/su_wiggles/Ux_file_single_v_wiggle.png
/Users/glennthompson/Downloads/Uz_file_single_v.su
  plotted /Users/glennthompson/Downloads/Uz_file_single_v.su to /Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/02_Modelling/Seismic/specfem2d/glenn/figures/explore_specfem_output/su_wiggles/Uz_file_single_v_wiggle.png


,file,byteorder,n_traces,figure_file
0,/Users/glennthompson/Downloads/Ux_file_single_...,<,299,/Users/glennthompson/Library/CloudStorage/Box-...
1,/Users/glennthompson/Downloads/Uz_file_single_...,<,299,/Users/glennthompson/Library/CloudStorage/Box-...


## 7. Difference two SEG-Y model gathers

This section reads two converted SEG-Y gathers, matches the common trace/sample dimensions, computes `A - B`, and plots the two inputs plus the difference.


In [14]:
def stream_to_gather_arrays(
    st,
    receiver_spacing_m=1.0,
    first_receiver_x_m=0.0,
):
    """Convert a single-component Stream to time, data, receiver_x arrays."""
    st = st.copy()
    st.sort(keys=["station", "channel"])

    if len(st) == 0:
        raise ValueError("Empty Stream")

    npts = min(tr.stats.npts for tr in st)
    dt = float(st[0].stats.delta)

    data = np.vstack([tr.data[:npts].astype(float) for tr in st])
    time = np.arange(npts, dtype=float) * dt
    receiver_x_m = first_receiver_x_m + np.arange(len(st), dtype=float) * receiver_spacing_m

    return time, data, receiver_x_m


def difference_segy_gathers(
    segy_a,
    segy_b,
    receiver_spacing_m=1.0,
    first_receiver_x_m=0.0,
):
    """Read two SEG-Y gathers and return matched A, B, A-B arrays."""
    st_a = obspy.read(str(segy_a), format="SEGY", unpack_trace_headers=True)
    st_b = obspy.read(str(segy_b), format="SEGY", unpack_trace_headers=True)

    time_a, data_a, rx_a = stream_to_gather_arrays(
        st_a,
        receiver_spacing_m=receiver_spacing_m,
        first_receiver_x_m=first_receiver_x_m,
    )
    time_b, data_b, rx_b = stream_to_gather_arrays(
        st_b,
        receiver_spacing_m=receiver_spacing_m,
        first_receiver_x_m=first_receiver_x_m,
    )

    ntr = min(data_a.shape[0], data_b.shape[0])
    npts = min(data_a.shape[1], data_b.shape[1])

    time = time_a[:npts]
    receiver_x_m = rx_a[:ntr]
    data_a = data_a[:ntr, :npts]
    data_b = data_b[:ntr, :npts]
    diff = data_a - data_b

    return time, data_a, data_b, diff, receiver_x_m


def plot_model_difference_from_segy(
    model_a,
    model_b,
    component="BXZ",
    segy_dir=segy_out_dir,
    source_x_m=0.0,
    receiver_spacing_m=1.0,
    first_receiver_x_m=0.0,
    tmin=0.0,
    tmax=0.3,
    omin=None,
    omax=None,
    clip_percentile=98,
    outfile=None,
):
    """Difference two converted model SEG-Y gathers and plot A, B, and A-B."""
    segy_dir = Path(segy_dir)
    segy_a = segy_dir / f"{model_a}_{component}.segy"
    segy_b = segy_dir / f"{model_b}_{component}.segy"

    if not segy_a.exists():
        raise FileNotFoundError(segy_a)
    if not segy_b.exists():
        raise FileNotFoundError(segy_b)

    time, data_a, data_b, diff, receiver_x_m = difference_segy_gathers(
        segy_a,
        segy_b,
        receiver_spacing_m=receiver_spacing_m,
        first_receiver_x_m=first_receiver_x_m,
    )

    if outfile is None:
        outfile = diff_fig_dir / f"{model_a}_minus_{model_b}_{component}.png"

    fig = plot_difference_gathers(
        time=time,
        data_a=data_a,
        data_b=data_b,
        receiver_x_m=receiver_x_m,
        source_x_m=source_x_m,
        label_a=model_a,
        label_b=model_b,
        title=f"{model_a} - {model_b}, {component}",
        tmin=tmin,
        tmax=tmax,
        omin=omin,
        omax=omax,
        clip_percentile=clip_percentile,
        outfile=outfile,
    )
    print(f"  plotted difference between {model_a} and {model_b} to {outfile}")

    return fig, diff


In [15]:
# Example model difference. Edit model names as needed.
# This assumes the batch conversion above has already written Mod12_BXZ.segy and Mod13_BXZ.segy.

fig, diff = plot_model_difference_from_segy(
    model_a="Mod12",
    model_b="Mod13",
    component="BXZ",
    source_x_m=source_x_m,
    receiver_spacing_m=receiver_spacing_m,
    first_receiver_x_m=first_receiver_x_m,
    tmin=0.0,
    tmax=0.3,
)


  plotted difference between Mod12 and Mod13 to /Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/02_Modelling/Seismic/specfem2d/glenn/figures/specfem_model_differences/Mod12_minus_Mod13_BXZ.png


## 8. Practical notes

- `normalize=False` preserves relative amplitudes between traces and is usually best for model comparison.
- `normalize=True` is useful for inspecting waveform shape when amplitudes vary strongly with offset.
- SU loading is faster when the files are valid, but some `.su` files may have headers ObsPy cannot parse. Those automatically fall back to SEM ASCII when available.
- SEG-Y files written here are intermediate QC products. They carry simple receiver/source x geometry based on the spacing parameters above; update those values when the exact survey geometry is known.
